# B₀ naive baseline sizing

B₀ is one complete, uncompressed flight-sensor frame. The calculation is **ceil(width × height × Σ band bit depths ÷ 8)**, plus any explicitly configured metadata allowance. It deliberately does not use the variable dimensions of the RGB training photographs.

In [1]:
from copy import deepcopy
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.baseline import compute_baseline, load_sensor_config

CONFIG_PATH = ROOT / 'configs' / 'sensor_config.yaml'
config = load_sensor_config(CONFIG_PATH)
baseline = compute_baseline(config)
baseline

{'name': 'Flight LWIR Boson 640 baseline',
 'platform': 'Conceptual cubesat payload (FLIR Boson 640-class)',
 'gsd_m': None,
 'frame_width_px': 640,
 'frame_height_px': 512,
 'bands': [{'name': 'LWIR',
   'wavelength_range_um': [7.5, 13.5],
   'bit_depth': 16,
   'purpose': 'primary_detection'}],
 'band_count': 1,
 'compression_ratio': 1.0,
 'metadata_bytes_per_frame': 0,
 'frames_per_day': 1,
 'pixels_per_frame': 327680,
 'bits_per_pixel_total': 16,
 'bits_per_frame': 5242880,
 'bytes_per_frame_raw': 655360,
 'bytes_per_frame_comp': 655360,
 'bytes_per_frame_total': 655360,
 'bytes_per_day': 655360}

In [2]:
band_depths = ' + '.join(f"{band['name']}: {band['bit_depth']} bits" for band in baseline['bands'])
print(f"pixels_per_frame = {baseline['frame_width_px']:,} width × {baseline['frame_height_px']:,} height = {baseline['pixels_per_frame']:,} pixels/frame")
print(f"bits_per_pixel_total = {band_depths} = {baseline['bits_per_pixel_total']:,} bits/pixel")
print(f"bits_per_frame = {baseline['pixels_per_frame']:,} pixels/frame × {baseline['bits_per_pixel_total']:,} bits/pixel = {baseline['bits_per_frame']:,} bits/frame")
print(f"bytes_per_frame_raw = ceil({baseline['bits_per_frame']:,} bits/frame ÷ 8 bits/byte) = {baseline['bytes_per_frame_raw']:,.0f} bytes/frame")
print(f"bytes_per_frame_comp = {baseline['bytes_per_frame_raw']:,.0f} bytes/frame ÷ {baseline['compression_ratio']:g} = {baseline['bytes_per_frame_comp']:,.0f} bytes/frame")
print(f"B0 = {baseline['bytes_per_frame_comp']:,.0f} payload bytes + {baseline['metadata_bytes_per_frame']:,} metadata bytes = {baseline['bytes_per_frame_total']:,.0f} bytes/frame")
print(f"bytes_per_day = {baseline['bytes_per_frame_total']:,.0f} bytes/frame × {baseline['frames_per_day']} frames/day = {baseline['bytes_per_day']:,.0f} bytes/day")
print()
print(f"B0 per frame: {baseline['bytes_per_frame_total'] / 1e6:,.3f} MB ({baseline['bytes_per_frame_total'] / 2**20:,.3f} MiB)")
print(f"B0 per day:   {baseline['bytes_per_day'] / 1e9:,.3f} GB ({baseline['bytes_per_day'] / 2**30:,.3f} GiB)")

pixels_per_frame = 640 width × 512 height = 327,680 pixels/frame
bits_per_pixel_total = LWIR: 16 bits = 16 bits/pixel
bits_per_frame = 327,680 pixels/frame × 16 bits/pixel = 5,242,880 bits/frame
bytes_per_frame_raw = ceil(5,242,880 bits/frame ÷ 8 bits/byte) = 655,360 bytes/frame
bytes_per_frame_comp = 655,360 bytes/frame ÷ 1 = 655,360 bytes/frame
B0 = 655,360 payload bytes + 0 metadata bytes = 655,360 bytes/frame
bytes_per_day = 655,360 bytes/frame × 1 frames/day = 655,360 bytes/day

B0 per frame: 0.655 MB (0.625 MiB)
B0 per day:   0.001 GB (0.001 GiB)


In [3]:
# The dataset is an RGB, low-fidelity training proxy. Its dimensions inform
# preprocessing, not the flight-frame B0 calculation above.
dataset = config['training_dataset_reference']
display(pd.DataFrame([
    ('Average', dataset['average_width_px'], dataset['average_height_px']),
    ('Minimum', dataset['minimum_width_px'], dataset['minimum_height_px']),
    ('Maximum', dataset['maximum_width_px'], dataset['maximum_height_px']),
    ('Standard deviation', dataset['width_stddev_px'], dataset['height_stddev_px']),
], columns=['Dataset statistic', 'Width (px)', 'Height (px)']))
print(dataset['note'])

,Dataset statistic,Width (px),Height (px)
0,Average,4057.00,3155.0
1,Minimum,153.00,206.0
2,Maximum,19699.00,8974.0
3,Standard deviation,1867.47,1388.6


These variable RGB image dimensions are useful for training-data loading and preprocessing only. They must not be used to estimate flight telemetry or make flight-representative performance claims.


In [4]:
# One-factor-at-a-time sensitivity. All candidates remain a single LWIR band.
def scenario(**overrides):
    candidate = deepcopy(config)
    candidate.update(overrides)
    return compute_baseline(candidate)

rows = []
for bit_depth in [8, 12, 16]:
    bands = [{**band, 'bit_depth': bit_depth} for band in config['bands']]
    rows.append(('LWIR bit depth', f'{bit_depth} bits/pixel', scenario(bands=bands)))
for ratio in [1, 2, 4]:
    rows.append(('compression ratio', f'{ratio}:1', scenario(compression_ratio=ratio)))
for width, height in [(320, 256), (640, 512), (1280, 1024)]:
    rows.append(('frame dimensions', f'{width} × {height} px', scenario(frame_width_px=width, frame_height_px=height)))

sensitivity = pd.DataFrame([
    {
        'Setting varied': key,
        'Value': value,
        'B0 (MB/frame)': result['bytes_per_frame_total'] / 1e6,
        'B0 (MiB/frame)': result['bytes_per_frame_total'] / 2**20,
    }
    for key, value, result in rows
])
display(sensitivity.style.format({'B0 (MB/frame)': '{:,.3f}', 'B0 (MiB/frame)': '{:,.3f}'}))

,Setting varied,Value,B0 (MB/frame),B0 (MiB/frame)
0,LWIR bit depth,8 bits/pixel,0.328,0.312
1,LWIR bit depth,12 bits/pixel,0.492,0.469
2,LWIR bit depth,16 bits/pixel,0.655,0.625
3,compression ratio,1:1,0.655,0.625
4,compression ratio,2:1,0.328,0.312
5,compression ratio,4:1,0.164,0.156
6,frame dimensions,320 × 256 px,0.164,0.156
7,frame dimensions,640 × 512 px,0.655,0.625
8,frame dimensions,1280 × 1024 px,2.621,2.500


In [5]:
RESULTS_PATH = ROOT / 'results' / 'baseline.json'
RESULTS_PATH.parent.mkdir(exist_ok=True)
with RESULTS_PATH.open('w', encoding='utf-8') as result_file:
    json.dump(baseline, result_file, indent=2)
print(f'Saved baseline result to {RESULTS_PATH}')

Saved baseline result to C:\Users\User\Desktop\wildfire\results\baseline.json


With the default flight sensor configuration, the naive baseline is **655,360 bytes per frame** (0.655 MB, 0.625 MiB): 640 × 512 pixels × one 16-bit LWIR band ÷ 8. At the temporary one-frame-per-day planning rate, that is also 655,360 bytes/day. The RGB training-dataset image-size statistics are intentionally excluded from this result.